# Solving Airy's Equation: Numerical Methods vs. Neural Networks

This notebook solves the **1-D Airy equation** (linearized KdV) using two approaches:

1. **Numerical Methods** — Explicit (RK4 + Centered Finite Differences) and Spectral Crank–Nicolson (FFT)
2. **Neural Network** — Physics-Informed Neural Network (PINN) via PyTorch

---

## Problem Statement

Airy's equation (as a PDE) is the simplest **purely dispersive** evolution equation:

$$\frac{\partial u}{\partial t} + \frac{\partial^3 u}{\partial x^3} = 0, \qquad x \in [0, 2\pi],\; t \in [0, T]$$

It arises as the **linear part** of the Korteweg–de Vries (KdV) equation $u_t + uu_x + u_{xxx} = 0$, and it is also connected to the classical Airy ODE $y'' - xy = 0$ through the fundamental solution (the Airy function $\mathrm{Ai}$).

Key properties:
- **Dispersive, not diffusive** — waves of different wavelengths travel at different speeds, but **no amplitude decay**.
- **Conservation of** $L^2$ **norm**: $\displaystyle\frac{d}{dt}\int_0^{2\pi} u^2\,dx = 0$ (energy is preserved exactly).
- **Dispersion relation**: substituting $u = e^{i(kx - \omega t)}$ yields $\omega = -k^3$, so the **phase velocity** is $v_p = \omega / k = -k^2$ — higher wavenumbers travel faster (to the left).

**Boundary conditions** — periodic:

$$u(0, t) = u(2\pi, t), \qquad u_x(0, t) = u_x(2\pi, t), \qquad u_{xx}(0, t) = u_{xx}(2\pi, t)$$

**Initial condition** — two-mode superposition to demonstrate dispersion:

$$u(x, 0) = \sin(x) + \tfrac{1}{2}\sin(2x)$$

**Exact solution** — each Fourier mode evolves independently:

$$u^*(x, t) = \sin(x + t) + \tfrac{1}{2}\sin\!\bigl(2(x + 4t)\bigr)$$

| Mode | $k$ | $\omega = -k^3$ | Phase velocity $v_p = -k^2$ | At $t = 1$: shift |
|------|-----|----------------|---------------------------|-------------------|
| 1    | 1   | $-1$           | $-1$ (left, speed 1)      | $1$ rad           |
| 2    | 2   | $-8$           | $-4$ (left, speed 4)      | $4$ rad           |

The $k = 2$ mode travels **4× faster** than the $k = 1$ mode, causing the initially overlapping wave components to separate progressively — the hallmark of dispersion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
CMAP = "inferno"

# ---- Problem parameters ---------------------
L_DOM = 2 * np.pi   # spatial domain [0, 2π] (periodic)
T_END = 1.0         # final time

# Exact-solution derived quantities
E0 = 0.5 * (np.pi + 0.25 * np.pi)   # ½ ∫₀²π u₀² dx = π/2 + π/8 = 5π/8

In [ ]:
def u_init(x):
    """Initial condition: two-mode superposition."""
    return np.sin(x) + 0.5 * np.sin(2 * x)


def u_exact(x, t):
    """Exact solution: each mode travels at its own phase speed."""
    return np.sin(x + t) + 0.5 * np.sin(2 * (x + 4 * t))


def periodic_extend(x, u, L):
    """Append the first point at x = L for clean periodic plots."""
    return np.append(x, L), np.append(u, u[0])


# Quick preview — exact solution at several times
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
x_plot = np.linspace(0, L_DOM, 500)
for ax, t_snap in zip(axes, [0.0, T_END / 3, 2 * T_END / 3, T_END]):
    ax.plot(x_plot, u_exact(x_plot, t_snap), "k-", lw=2)
    ax.fill_between(x_plot, u_exact(x_plot, t_snap), alpha=0.25, color="steelblue")
    ax.set_title(f"$t = {t_snap:.3f}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.8, 1.8)
    ax.grid(alpha=0.3)

plt.suptitle("Exact Solution — Airy Equation (Dispersive Two-Mode Wave)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Domain: [0, 2π],  T = {T_END}")
print(f"Mode k=1: phase speed |v_p| = 1,   shift at T = {T_END:.1f} rad")
print(f"Mode k=2: phase speed |v_p| = 4,   shift at T = {4*T_END:.1f} rad")
print(f"L² energy E₀ = 5π/8 = {E0:.6f}  (conserved for all t)")

---

## Part 1 — Numerical Methods

### 1-A. Explicit: RK4 + Centered Finite Differences

We discretize $u_t = -u_{xxx}$ as a **method of lines**: spatial derivatives are approximated on a uniform periodic grid, then integrated in time with classical **fourth-order Runge–Kutta** (RK4).

The centered, second-order stencil for the third derivative is:

$$u_{xxx}(x_j) \approx \frac{u_{j+2} - 2u_{j+1} + 2u_{j-1} - u_{j-2}}{2\,\Delta x^3}$$

Periodic boundaries are handled by circular indexing (no ghost cells needed).

**Stability**: RK4's stability region barely touches the imaginary axis at $\pm i\sqrt{8}$. Since the eigenvalues of the skew-symmetric $D_{xxx}$ operator are purely imaginary with maximum magnitude $\sim 3\sqrt{3}/(2\,\Delta x^3)$, the CFL condition is:

$$\Delta t \leq \frac{4\sqrt{2}}{3\sqrt{3}}\,\Delta x^3 \approx 1.089\,\Delta x^3$$

The **cubic** dependence on $\Delta x$ makes this scheme expensive for fine grids — a hallmark of third-order PDEs.

In [ ]:
def solve_rk4_fd(N=128, T=T_END):
    """
    Explicit RK4 + centred FD for u_t + u_xxx = 0 with periodic BCs.

    Parameters
    ----------
    N : int   Number of grid points.
    T : float Final time.

    Returns
    -------
    x, u_hist, t_hist
    """
    dx = L_DOM / N
    x  = np.arange(N) * dx        # periodic: [0, dx, ..., (N-1)·dx)

    # CFL:  Δt ≤ ~1.089 · dx³
    dt_max = 1.0 * dx**3           # conservative factor (< 1.089)
    Nt     = max(int(np.ceil(T / dt_max)), 100)
    dt     = T / Nt

    print(f"RK4+FD  — N={N}, dx={dx:.5f}, Nt={Nt}, dt={dt:.2e}, "
          f"CFL dt/dx³={dt / dx**3:.3f}")

    def rhs(u):
        """−u_xxx via centred 2nd-order FD with periodic wrap."""
        return -(np.roll(u, -2) - 2 * np.roll(u, -1)
                 + 2 * np.roll(u, 1) - np.roll(u, 2)) / (2 * dx**3)

    u = u_init(x)
    snap_steps = {0, Nt // 3, 2 * Nt // 3, Nt}
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_hist.append(u.copy())
            t_hist.append(n * dt)
        if n == Nt:
            break

        # Classical RK4
        k1 = rhs(u)
        k2 = rhs(u + 0.5 * dt * k1)
        k3 = rhs(u + 0.5 * dt * k2)
        k4 = rhs(u + dt * k3)
        u  = u + (dt / 6) * (k1 + 2 * k2 + 2 * k3 + k4)

    return x, u_hist, t_hist


x_rk, u_rk_hist, t_rk_hist = solve_rk4_fd(N=128)

# Error at final time
u_ref_rk = u_exact(x_rk, T_END)
err_rk   = np.abs(u_rk_hist[-1] - u_ref_rk)
print(f"Max absolute error  : {err_rk.max():.3e}")
print(f"Mean absolute error : {err_rk.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_rk_hist, t_rk_hist)):
    xp, up = periodic_extend(x_rk, u_snap,     L_DOM)
    _,  ur = periodic_extend(x_rk, u_exact(x_rk, t_snap), L_DOM)

    axes[0, col].plot(xp, ur, "k-", lw=2, label="Exact")
    axes[0, col].plot(xp, up, "b--", lw=1.5, label="RK4+FD")
    axes[0, col].set_title(f"$t = {t_snap:.3f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.8, 1.8)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_exact(x_rk, t_snap))
    xp_e, ep = periodic_extend(x_rk, err, L_DOM)
    axes[1, col].plot(xp_e, ep, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Explicit RK4 + Centred FD — Airy Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1-B. Implicit: Spectral Crank–Nicolson (FFT)

For periodic domains, the natural approach is to work in **Fourier space**.

The Fourier symbol of $\partial_{xxx}$ is $(ik)^3 = -ik^3$, so the PDE in Fourier space is:

$$\frac{d\hat{u}_k}{dt} = ik^3 \hat{u}_k$$

Applying the Crank–Nicolson (trapezoidal) rule:

$$\frac{\hat{u}_k^{n+1} - \hat{u}_k^n}{\Delta t} = \frac{ik^3}{2}\left(\hat{u}_k^{n+1} + \hat{u}_k^n\right)$$

$$\hat{u}_k^{n+1} = \underbrace{\frac{1 + \tfrac{i k^3 \Delta t}{2}}{1 - \tfrac{i k^3 \Delta t}{2}}}_{g_k}\;\hat{u}_k^n$$

Since numerator and denominator are complex conjugates, $|g_k| = 1$ for every mode — the scheme is:

- **Unconditionally stable** (no CFL constraint)
- **Exactly energy-preserving** ($\sum |\hat{u}_k|^2$ is constant)
- **Spectrally accurate** in space (machine-precision spatial derivatives)
- **Second-order accurate** in time: $\mathcal{O}(\Delta t^2)$

Each time step requires only **two FFTs** ($\mathcal{O}(N\log N)$) — no linear system solve.

In [ ]:
def solve_cn_fft(N=128, Nt=300, T=T_END):
    """
    Spectral Crank–Nicolson (FFT) for u_t + u_xxx = 0, periodic BCs.

    Parameters
    ----------
    N  : int   Grid points.
    Nt : int   Time steps.
    T  : float Final time.

    Returns
    -------
    x, u_hist, t_hist
    """
    dx = L_DOM / N
    x  = np.arange(N) * dx
    dt = T / Nt

    # Wavenumbers for periodic domain [0, 2π]
    k = np.fft.fftfreq(N, d=dx) * (2 * np.pi)      # 0, 1, ..., N/2, -N/2+1, ..., -1

    # CN amplification factor (|g_k| = 1 ∀k)
    g = (1 + 0.5j * k**3 * dt) / (1 - 0.5j * k**3 * dt)

    print(f"CN+FFT  — N={N}, Nt={Nt}, dt={dt:.5f}, "
          f"max ||g|-1| = {np.max(np.abs(np.abs(g) - 1)):.2e}")

    u_hat = np.fft.fft(u_init(x))

    snap_steps = {0, Nt // 3, 2 * Nt // 3, Nt}
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_hist.append(np.real(np.fft.ifft(u_hat)).copy())
            t_hist.append(n * dt)
        if n == Nt:
            break
        u_hat *= g

    return x, u_hist, t_hist


x_cn, u_cn_hist, t_cn_hist = solve_cn_fft(N=128, Nt=300)

u_ref_cn = u_exact(x_cn, T_END)
err_cn   = np.abs(u_cn_hist[-1] - u_ref_cn)
print(f"Max absolute error  : {err_cn.max():.3e}")
print(f"Mean absolute error : {err_cn.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_cn_hist, t_cn_hist)):
    xp, up = periodic_extend(x_cn, u_snap,     L_DOM)
    _,  ur = periodic_extend(x_cn, u_exact(x_cn, t_snap), L_DOM)

    axes[0, col].plot(xp, ur, "k-", lw=2, label="Exact")
    axes[0, col].plot(xp, up, "g--", lw=1.5, label="CN+FFT")
    axes[0, col].set_title(f"$t = {t_snap:.3f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.8, 1.8)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_exact(x_cn, t_snap))
    xp_e, ep = periodic_extend(x_cn, err, L_DOM)
    axes[1, col].plot(xp_e, ep, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Spectral Crank–Nicolson (FFT) — Airy Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Grid-refinement convergence study ----------------------------------------
N_list = [16, 32, 64, 128]
err_rk_conv, err_cn_conv = [], []

for N in N_list:
    # Explicit RK4+FD (CFL-limited)
    x_c, u_c, t_c = solve_rk4_fd(N, T=T_END)
    err_rk_conv.append(np.max(np.abs(u_c[-1] - u_exact(x_c, t_c[-1]))))

    # Spectral CN+FFT (Nt = 500 fixed)
    x_c2, u_c2, t_c2 = solve_cn_fft(N, Nt=500, T=T_END)
    err_cn_conv.append(np.max(np.abs(u_c2[-1] - u_exact(x_c2, t_c2[-1]))))

dh = [L_DOM / N for N in N_list]

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(dh, err_rk_conv, "bs-",  lw=1.5, label="Explicit (RK4+FD)")
ax.loglog(dh, err_cn_conv, "ro-",  lw=1.5, label="Spectral CN (FFT)")
ax.loglog(dh, [h**2 * err_rk_conv[0] / dh[0]**2 for h in dh],
          "b--", lw=1.0, label="$\\mathcal{O}(h^2)$")
ax.set_xlabel("Grid spacing $h = 2\\pi / N$")
ax.set_ylabel("Max absolute error at $t = T$")
ax.set_title("Convergence Study — Numerical Schemes")
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()
print("RK4+FD shows O(h²) convergence (spatial FD dominates).")
print("CN+FFT achieves spectral accuracy in space — error drops exponentially,")
print("saturating at the O(Δt²) temporal truncation floor.")

---

## Part 2 — Neural Network: Physics-Informed Neural Network (PINN)

### How It Works

A PINN learns $\hat{u}_\theta(x, t)$ by minimising:

$$\mathcal{L} = \underbrace{\mathcal{L}_{IC}}_{\text{initial condition}} + \underbrace{\mathcal{L}_{BC}}_{\text{periodic matching}} + \underbrace{\mathcal{L}_{PDE}}_{\text{physics residual}}$$

$$\mathcal{L}_{IC} = \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}(x_k, 0) - u_0(x_k)\Bigr)^2$$

$$\mathcal{L}_{BC} = \frac{1}{N_{BC}}\sum_k \left[\bigl(\hat{u}(0, t_k) - \hat{u}(2\pi, t_k)\bigr)^2 + \bigl(\hat{u}_x(0, t_k) - \hat{u}_x(2\pi, t_k)\bigr)^2\right]$$

$$\mathcal{L}_{PDE} = \frac{1}{N_f}\sum_k \left(\frac{\partial \hat{u}}{\partial t} + \frac{\partial^3 \hat{u}}{\partial x^3}\right)^2$$

**Key difference**: The PDE residual requires the **third spatial derivative** $u_{xxx}$, which needs three successive autograd passes through the network. Tanh activations ensure these derivatives exist and are smooth.

**Periodic BCs** are enforced as **soft constraints** via $\mathcal{L}_{BC}$: matching both $u$ and $u_x$ at $x = 0$ and $x = 2\pi$.

Training: **Adam** warm-up → **L-BFGS** fine-tuning.

In [ ]:
# -----------------------------------------------------------------
# Network Architecture
# -----------------------------------------------------------------
class AiryPINN(nn.Module):
    """Fully-connected PINN: (x, t) → u.
    Tanh activations ensure smooth third-order derivatives via autograd.
    """

    def __init__(self, hidden_layers=5, hidden_dim=80):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


def grad1(u, v):
    return torch.autograd.grad(
        u, v, grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]


# -----------------------------------------------------------------
# Collocation / IC / BC Points
# -----------------------------------------------------------------
N_IC  = 2000
N_BC  = 1000
N_INT = 10000

# IC points at t = 0
x_ic = (torch.rand(N_IC, 1) * L_DOM).requires_grad_(True)
t_ic = torch.zeros(N_IC, 1, requires_grad=True)
u_ic_target = torch.tensor(
    u_init(x_ic.detach().numpy()), dtype=torch.float32
)

# Periodic BC points (same t for left and right)
t_bc   = torch.rand(N_BC, 1) * T_END
x_bc_0 = torch.zeros(N_BC, 1, requires_grad=True)
x_bc_L = torch.full((N_BC, 1), L_DOM, requires_grad=True)

# Interior PDE collocation
x_int = (torch.rand(N_INT, 1) * L_DOM).requires_grad_(True)
t_int = (torch.rand(N_INT, 1) * T_END).requires_grad_(True)

mse = nn.MSELoss()

print(f"IC  points : {N_IC}")
print(f"BC  points : {2 * N_BC} (periodic matching)")
print(f"PDE points : {N_INT}")

In [ ]:
def compute_loss(model):
    # ---- IC loss ----
    u_pred_ic = model(x_ic, t_ic)
    loss_ic   = mse(u_pred_ic, u_ic_target)

    # ---- Periodic BC loss: u and u_x must match at x=0 and x=2π ----
    u_at_0  = model(x_bc_0, t_bc)
    u_at_L  = model(x_bc_L, t_bc)
    loss_bc = mse(u_at_0, u_at_L)

    ux_at_0 = grad1(u_at_0, x_bc_0)
    ux_at_L = grad1(u_at_L, x_bc_L)
    loss_bc = loss_bc + mse(ux_at_0, ux_at_L)

    # ---- PDE residual: u_t + u_xxx = 0 ----
    u_pred = model(x_int, t_int)
    u_t    = grad1(u_pred, t_int)
    u_x    = grad1(u_pred, x_int)
    u_xx   = grad1(u_x,   x_int)
    u_xxx  = grad1(u_xx,  x_int)
    residual = u_t + u_xxx
    loss_pde = mse(residual, torch.zeros_like(residual))

    return loss_ic + loss_bc + loss_pde, loss_ic, loss_bc, loss_pde


# ---- Phase 1: Adam --------------------------------
model    = AiryPINN(hidden_layers=5, hidden_dim=80)
opt_adam = optim.Adam(model.parameters(), lr=1e-3)
ADAM_EP  = 5000
history  = []

for ep in range(1, ADAM_EP + 1):
    opt_adam.zero_grad()
    loss, l_ic, l_bc, l_pde = compute_loss(model)
    loss.backward()
    opt_adam.step()
    history.append(loss.item())
    if ep % 1000 == 0:
        print(f"[Adam] Ep {ep:5d} | Loss {loss.item():.5f} | "
              f"IC {l_ic.item():.5f} | BC {l_bc.item():.5f} | PDE {l_pde.item():.5f}")

print("Adam phase done.")

In [ ]:
# ---- Phase 2: L-BFGS ----
opt_lbfgs = optim.LBFGS(
    model.parameters(),
    max_iter=500, tolerance_grad=1e-9, tolerance_change=1e-12,
    history_size=50, line_search_fn="strong_wolfe"
)


def closure():
    opt_lbfgs.zero_grad()
    loss, _, _, _ = compute_loss(model)
    loss.backward()
    history.append(loss.item())
    return loss


opt_lbfgs.step(closure)
final, _, _, _ = compute_loss(model)
print(f"L-BFGS done.  Final loss: {final.item():.6f}")

# Training loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(history, color="steelblue", lw=1.0)
ax.axvline(x=ADAM_EP, color="tomato", ls="--", lw=1.5, label="Adam → L-BFGS")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Loss (log scale)")
ax.set_title("PINN Training Loss — Airy Equation")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Evaluate PINN on a dense grid ----
model.eval()
Nev  = 500
x_ev = np.linspace(0, L_DOM, Nev, endpoint=False)


def pinn_predict(t_val):
    xt = torch.tensor(x_ev, dtype=torch.float32).unsqueeze(1)
    tt = torch.full((Nev, 1), t_val, dtype=torch.float32)
    with torch.no_grad():
        return model(xt, tt).numpy().ravel()


snap_times_pinn = [0.0, T_END / 3, 2 * T_END / 3, T_END]
U_pinn_snaps    = [pinn_predict(t) for t in snap_times_pinn]

U_ex_pinn = u_exact(x_ev, T_END)
err_pinn  = np.abs(U_pinn_snaps[-1] - U_ex_pinn)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (U_snap, t_snap) in enumerate(zip(U_pinn_snaps, snap_times_pinn)):
    u_ref = u_exact(x_ev, t_snap)
    xp, up = periodic_extend(x_ev, U_snap, L_DOM)
    _,  ur = periodic_extend(x_ev, u_ref,  L_DOM)

    axes[0, col].plot(xp, ur, "k-", lw=2, label="Exact")
    axes[0, col].plot(xp, up, "m--", lw=1.5, label="PINN")
    axes[0, col].set_title(f"$t = {t_snap:.3f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.8, 1.8)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(U_snap - u_ref)
    xp_e, ep = periodic_extend(x_ev, err, L_DOM)
    axes[1, col].plot(xp_e, ep, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Physics-Informed Neural Network — Airy Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"PINN — Max error at t=T: {err_pinn.max():.2e}  |  Mean error: {err_pinn.mean():.2e}")

---

## Part 3 — Side-by-Side Comparison

Solutions from all three methods at $t = T$ alongside the exact solution, plus a quantitative error summary.

In [ ]:
# Interpolate FD solutions onto the common evaluation grid

def periodic_interp(x_fd, u_fd, x_eval, L):
    """Linearly interpolate periodic data to a new grid."""
    x_ext = np.concatenate([x_fd - L, x_fd, x_fd + L])
    u_ext = np.concatenate([u_fd, u_fd, u_fd])
    return np.interp(x_eval, x_ext, u_ext)


u_rk_ev   = periodic_interp(x_rk, u_rk_hist[-1], x_ev, L_DOM)
u_cn_ev   = periodic_interp(x_cn, u_cn_hist[-1], x_ev, L_DOM)
u_pinn_ev = U_pinn_snaps[-1]

err_rk_ev   = np.abs(u_rk_ev   - U_ex_pinn)
err_cn_ev   = np.abs(u_cn_ev   - U_ex_pinn)
err_pinn_ev = np.abs(u_pinn_ev - U_ex_pinn)

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

labels = ["Explicit (RK4+FD)", "Spectral CN (FFT)", "PINN", "Exact"]
sols   = [u_rk_ev, u_cn_ev, u_pinn_ev, U_ex_pinn]
colors = ["steelblue", "seagreen", "mediumorchid", "black"]
errs   = [err_rk_ev, err_cn_ev, err_pinn_ev]
e_lbl  = ["RK4+FD error", "CN+FFT error", "PINN error"]

for col, (lbl, sol, c) in enumerate(zip(labels, sols, colors)):
    xp, sp = periodic_extend(x_ev, sol, L_DOM)
    ax = fig.add_subplot(gs[0, col])
    ax.plot(xp, sp, color=c, lw=2)
    ax.fill_between(xp, sp, alpha=0.2, color=c)
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.8, 1.8)
    ax.grid(alpha=0.3)

for col, (lbl, err) in enumerate(zip(e_lbl, errs)):
    xp, ep = periodic_extend(x_ev, err, L_DOM)
    ax = fig.add_subplot(gs[1, col])
    ax.plot(xp, ep, "r-", lw=1.5)
    ax.set_title(f"{lbl}  (max {err.max():.1e})", fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("|error|")
    ax.grid(alpha=0.3)

# Overlay
xp, ue = periodic_extend(x_ev, U_ex_pinn, L_DOM)
_, u1   = periodic_extend(x_ev, u_rk_ev,   L_DOM)
_, u2   = periodic_extend(x_ev, u_cn_ev,   L_DOM)
_, u3   = periodic_extend(x_ev, u_pinn_ev, L_DOM)

ax_ov = fig.add_subplot(gs[1, 3])
ax_ov.plot(xp, ue, "k-",  lw=2,   label="Exact")
ax_ov.plot(xp, u1, "b--", lw=1.5, label="RK4+FD")
ax_ov.plot(xp, u2, "g:",  lw=1.5, label="CN+FFT")
ax_ov.plot(xp, u3, "m-.", lw=1.5, label="PINN")
ax_ov.set_title("$t = T$ overlay", fontsize=10)
ax_ov.set_xlabel("$x$")
ax_ov.set_ylabel("$u$")
ax_ov.legend(fontsize=8)
ax_ov.grid(alpha=0.3)

fig.suptitle(f"Final Comparison at $t = T = {T_END}$ — Airy Equation",
             fontsize=13, fontweight="bold")
plt.show()

In [ ]:
# ---- Quantitative error summary ----
def l2_norm(f_arr, x_arr):
    return np.sqrt(np.trapezoid(f_arr**2, x_arr))


# L² energy conservation check
def compute_l2_energy(u_arr, x_arr):
    """½ ∫ u² dx via trapezoidal rule on periodic grid."""
    xp, up = periodic_extend(x_arr, u_arr, L_DOM)
    return 0.5 * np.trapezoid(up**2, xp)


E_rk   = compute_l2_energy(u_rk_hist[-1], x_rk)
E_cn   = compute_l2_energy(u_cn_hist[-1], x_cn)
E_pinn = compute_l2_energy(U_pinn_snaps[-1], x_ev)

print("=" * 76)
print(f"{'Method':<22} {'Max error':>12} {'Mean error':>12} {'L² error':>12} {'E(T)':>10} {'|ΔE/E₀|':>10}")
print("-" * 76)
print(f"{'Exact':22} {'—':>12} {'—':>12} {'—':>12} {E0:>10.6f} {'0':>10}")
print(f"{'RK4 + FD':22} {err_rk_ev.max():>12.3e} {err_rk_ev.mean():>12.3e} "
      f"{l2_norm(err_rk_ev, x_ev):>12.3e} {E_rk:>10.6f} {abs(E_rk - E0) / E0:>10.2e}")
print(f"{'Spectral CN (FFT)':22} {err_cn_ev.max():>12.3e} {err_cn_ev.mean():>12.3e} "
      f"{l2_norm(err_cn_ev, x_ev):>12.3e} {E_cn:>10.6f} {abs(E_cn - E0) / E0:>10.2e}")
print(f"{'PINN':22} {err_pinn_ev.max():>12.3e} {err_pinn_ev.mean():>12.3e} "
      f"{l2_norm(err_pinn_ev, x_ev):>12.3e} {E_pinn:>10.6f} {abs(E_pinn - E0) / E0:>10.2e}")
print("=" * 76)
print(f"\nExact L² energy: E₀ = 5π/8 = {E0:.6f}")
print("The Airy equation conserves L² energy exactly — numerical deviations")
print("indicate the scheme's energy fidelity.")

---

## Summary

### About Airy's Equation

The Airy PDE $u_t + u_{xxx} = 0$ is the prototypical **purely dispersive** equation — the linear skeleton of the KdV equation and a fundamental model in wave physics:

- **No dissipation**: the $L^2$ norm $\int u^2\,dx$ is conserved exactly (unlike the telegraph or heat equations).
- **Frequency-dependent propagation**: the dispersion relation $\omega = -k^3$ means higher-frequency components travel faster. An initially localized pulse will **spread and oscillate** — the Airy function pattern emerges from the Green's function.
- **Third-order operator**: the odd-order spatial derivative $u_{xxx}$ makes the operator *skew-adjoint*, which is responsible for both the energy conservation and the purely imaginary spectrum.
- **CFL challenge**: explicit schemes require $\Delta t \propto \Delta x^3$, making them costly for fine grids.

### Method Comparison

| Aspect | Explicit (RK4 + FD) | Spectral CN (FFT) | PINN |
|--------|--------------------|--------------------|------|
| **Spatial discretization** | Centred FD, $\mathcal{O}(h^2)$ | FFT — spectral accuracy | Continuous (meshfree) |
| **Time integration** | RK4, $\mathcal{O}(\Delta t^4)$ | CN, $\mathcal{O}(\Delta t^2)$ | N/A (learned jointly) |
| **Stability** | CFL: $\Delta t \lesssim h^3$ | Unconditional | Unconditional |
| **Energy conservation** | Approximate ($|g| \approx 1$) | Exact ($|g_k| = 1$) | Approximate (not enforced) |
| **Cost per step** | $\mathcal{O}(N)$ | $\mathcal{O}(N\log N)$ (2 FFTs) | N/A |
| **Boundary type** | Periodic (circular index) | Periodic (inherent in DFT) | Soft-constrained |
| **Best for** | Moderate grids, prototyping | Production — fast and accurate | Inverse / parametric problems |

### Key Observations

- **Third-order PDEs** are numerically challenging: the $\Delta t \sim h^3$ CFL for explicit schemes is far more restrictive than the familiar $\Delta t \sim h$ or $\Delta t \sim h^2$ conditions for wave and diffusion equations. This strongly motivates **implicit or spectral methods**.
- The **FFT-based Crank–Nicolson** scheme is ideally suited for Airy's equation on periodic domains: it achieves spectral accuracy in space, exact energy conservation, and unconditional stability — all with $\mathcal{O}(N\log N)$ cost per step and zero linear system solves.
- The **PINN** handles the third derivative $u_{xxx}$ through three successive autograd passes. The Tanh activation ensures the derivatives exist and are smooth. Periodic BCs are enforced softly by penalising mismatches at $x = 0$ and $x = 2\pi$ — a natural approach but less precise than the FFT's intrinsic periodicity.
- **Dispersion is clearly visible**: the two-mode initial condition separates progressively, with the $k = 2$ component racing ahead at 4× the speed of the $k = 1$ mode. All three methods capture this dispersive separation accurately.